# Activity 2: Horospheres and a Two-Layer HBNN

This notebook follows two tasks:

1. visualize horospheres in the Poincaré and Lorentz models.
2. train a two-layer HBNN, consisting of a Busemann fully connected (BFC) layer and a Busemann multinomial logistic regression (BMLR) layer, on a synthetic classification dataset.

## Mathematical preliminaries

For curvature $K<0$, we use the Poincaré and Lorentz models.

<p align="center">
  <img src="https://raw.githubusercontent.com/GitZH-Chen/MLSS-RDL-Tutorial/main/assets/hbnn_models.svg" alt="Definitions of the Poincaré ball, Lorentz model, Lorentz inner product, and hyperbolic space" width="550">
</p>

Let $\gamma$ be a unit-speed geodesic ray. Its **Busemann function** is shown below. In Euclidean space, it reduces to the inner product up to sign.

<p align="center">
  <img src="https://raw.githubusercontent.com/GitZH-Chen/MLSS-RDL-Tutorial/main/assets/hbnn_busemann_definition.svg" alt="Busemann function and its Euclidean reduction" width="500">
</p>

For a unit direction $v\in\mathbb{S}^{n-1}$, the closed forms used in HBNN are

<p align="center">
  <img src="https://raw.githubusercontent.com/GitZH-Chen/MLSS-RDL-Tutorial/main/assets/hbnn_busemann_closed_forms.svg" alt="Closed forms of the Busemann function in the Poincaré and Lorentz models" width="465">
</p>

A **horosphere** is a level set.

<p align="center">
  <img src="https://raw.githubusercontent.com/GitZH-Chen/MLSS-RDL-Tutorial/main/assets/hbnn_horosphere.svg" alt="Definition of a horosphere as a Busemann level set" width="290">
</p>

The correspondence used by HBNN is summarized below.

<p align="center">
  <img src="https://raw.githubusercontent.com/GitZH-Chen/MLSS-RDL-Tutorial/main/assets/hbnn_euclidean_hyperbolic_table.svg" alt="Euclidean and hyperbolic correspondence used by HBNN" width="1000">
</p>

## References

- Ziheng Chen, Bernhard Schölkopf, and Nicu Sebe. **Hyperbolic Busemann Neural Networks.** CVPR 2026. [Paper](https://arxiv.org/abs/2602.18858) · [Code](https://github.com/GitZH-Chen/HBNN)

## Setup

The notebook downloads the public HBNN implementation at its tested revision.

In [ ]:
# Install the geometry dependency required by HBNN.
%pip -q install geoopt==0.5.1

!test -d /content/mlss_hbnn || git clone -q https://github.com/GitZH-Chen/HBNN.git /content/mlss_hbnn
!git -C /content/mlss_hbnn checkout -q d5c79c8eed36a0b7c2f15e9a8fbcd0318216e5b0

In [ ]:
import random
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

warnings.filterwarnings("ignore", category=SyntaxWarning)
sys.path.insert(0, "/content/mlss_hbnn")

from lib.bnn.BFC import BFC
from lib.bnn.BMLR import BMLR
from lib.bnn.Geometry import Stereographic
from lib.bnn.evaluation.vis_horosphere import main as plot_horospheres


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(7)
print("PyTorch:", torch.__version__)

## Task 1: Visualize horospheres

We fix $K=-1$ and $v=(1,0)$. Each curve is a level set $B^v(x)=\tau$.

**Predict:** How should horospheres with the same direction differ as $\tau$ changes?

In [ ]:
# Plot the Poincaré and Lorentz horospheres.
plot_horospheres()

**Interpret:** In the Poincaré ball, the horospheres are Euclidean circles tangent to the boundary at the same ideal point. In the Lorentz model, they are intersections of the hyperboloid with parallel affine hyperplanes.

## Task 2: Train a two-layer HBNN

The learnable network is

<p align="center">
  <img src="https://raw.githubusercontent.com/GitZH-Chen/MLSS-RDL-Tutorial/main/assets/hbnn_network_table.svg" alt="Two-layer HBNN architecture" width="600">
</p>

The exponential map embeds the input. The two trainable layers are BFC and BMLR.

### Generate a synthetic dataset

In [ ]:
centers = torch.tensor([[-1.5, -0.8], [1.5, -0.8], [0.0, 1.5]])
features = torch.cat([center + 0.55 * torch.randn(120, 2) for center in centers])
labels = torch.arange(3).repeat_interleave(120)

permutation = torch.randperm(len(features))
train_ids = permutation[:280]
validation_ids = permutation[280:]
x_train, y_train = features[train_ids], labels[train_ids]
x_validation, y_validation = features[validation_ids], labels[validation_ids]

plt.figure(figsize=(5, 4))
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, cmap="viridis", s=22)
plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Synthetic training data")
plt.show()

### Define the network

**Predict:** For a batch of 8 samples and 3 classes, what is the shape of the BMLR output?

In [ ]:
class TwoLayerHBNN(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=8, n_classes=3, curvature=-1.0):
        super().__init__()
        self.manifold = Stereographic(K=curvature)
        self.hidden = BFC(
            in_dim=input_dim,
            out_dim=hidden_dim,
            metric="poincare",
            K=curvature,
            act="relu",
        )
        self.classifier = BMLR(
            n_classes=n_classes,
            dim=hidden_dim,
            metric="poincare",
            K=curvature,
        )

    def forward(self, x):
        x_hyperbolic = self.manifold.exp0(0.25 * x)
        hidden_hyperbolic = self.hidden(x_hyperbolic)
        return self.classifier(hidden_hyperbolic)


model = TwoLayerHBNN()
print(model)

### Train

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
criterion = nn.CrossEntropyLoss()
loss_history = []
validation_accuracy_history = []

for epoch in range(151):
    model.train()
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        validation_predictions = model(x_validation).argmax(dim=1)
        validation_accuracy = (
            validation_predictions == y_validation
        ).float().mean().item()

    loss_history.append(loss.item())
    validation_accuracy_history.append(validation_accuracy)

    if epoch % 30 == 0:
        print(
            f"Epoch {epoch:3d} | loss {loss.item():.4f} | "
            f"validation accuracy {validation_accuracy:.3f}"
        )

### Evaluate and visualize

In [ ]:
model.eval()
with torch.no_grad():
    train_accuracy = (
        model(x_train).argmax(dim=1) == y_train
    ).float().mean().item()
    validation_accuracy = (
        model(x_validation).argmax(dim=1) == y_validation
    ).float().mean().item()

grid_x, grid_y = torch.meshgrid(
    torch.linspace(-3.0, 3.0, 180),
    torch.linspace(-3.0, 3.0, 180),
    indexing="xy",
)
grid = torch.stack([grid_x.flatten(), grid_y.flatten()], dim=1)

with torch.no_grad():
    grid_predictions = model(grid).argmax(dim=1).reshape(grid_x.shape)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].contourf(grid_x, grid_y, grid_predictions, alpha=0.25, cmap="viridis")
axes[0].scatter(
    x_validation[:, 0],
    x_validation[:, 1],
    c=y_validation,
    cmap="viridis",
    edgecolor="black",
    s=28,
)
axes[0].set_title("HBNN decision regions")
axes[1].plot(loss_history)
axes[1].set(xlabel="Epoch", ylabel="Cross-entropy", title="Training loss")
axes[2].plot(validation_accuracy_history)
axes[2].set(
    xlabel="Epoch",
    ylabel="Accuracy",
    ylim=(0, 1.02),
    title="Validation accuracy",
)
plt.tight_layout()
plt.show()

print(f"Train accuracy: {train_accuracy:.3f}")
print(f"Validation accuracy: {validation_accuracy:.3f}")

## Takeaways

- Busemann functions generalize Euclidean inner products.
- Horospheres are the hyperbolic counterparts of Euclidean hyperplanes.
- BFC maps hyperbolic features to hyperbolic features.
- BMLR maps hyperbolic features to class logits.
- Both layers can be trained end to end with ordinary PyTorch optimization.